# Analys av hälsostudie – Del 2

Här bygger vi vidare på Del 1; nu med funktioner, klasser och linjär regression.
Vi flyttar kod till `src/`, gör en klass och testar enkel regression.

In [1]:
# Importerar vår nya klass – enkel och ren!
# Källa: Videor (moduler), (klasser)
from src.health_analyzer import HealthAnalyzer

# Skapar objekt – laddar data automatiskt och visar head!
analyzer = HealthAnalyzer("data/dataset.csv")
analyzer.df.head()

Data laddad: 800 rader


,id,age,sex,height,weight,systolic_bp,cholesterol,smoker,disease
0,1,57,F,168.9,65.8,141.8,4.58,No,0
1,2,47,M,180.4,95.9,144.8,5.18,Yes,0
2,3,59,F,169.9,82.2,151.7,6.16,No,0
3,4,72,M,157.7,93.1,151.0,6.63,No,0
4,5,46,M,192.6,104.1,144.1,5.21,No,0


## Beskrivande statistik – vad säger siffrorna.

Nu räknar vi ut medel, median, min och max för ålder, vikt, längd, blodtryck och kolesterol.  
Detta ger oss en snabb bild av hur deltagaren ser ut

In [ ]:
# Importerar vår funktion från src
# Källa: Video (moduler)

from src.stats_utils import calculate_descriptive_stats

columns = ["age", "weight", "height", "systolic_bp", "cholesterol"]

# Automatisk sammanfattning

print("Automatisk sammanfattning från Pandas:")
print(df[columns].describe())


# stats = calculate_descriptive_stats(df, columns)
# print("\n Manuellt räknat:")
# for col, values in stats.items():
#     print(f"{col}: medel = {values['mean']:.2f}, "
#           f"median = {values['median']:.2f}, "
#           f"min = {values['min']:.2f}, max = {values['max']:.2f}")

## Grafer – vad ser vi i datan?

Ritar tre enkla grafer för att få en visuell känsla för hälsostudien:  
- Hur ser blodtrycket ut? (histogram)  
- Finns skillnad i vikt mellan män och kvinnor? (boxplot)  
- Hur många röker egentligen? (stapeldiagram)  

In [ ]:
# Ritar histogram för att se hur blodtrycket är fördelat
# är det normalfördelat?
# Videor - Matplotlib

plt.figure(figsize=(8, 5))
plt.hist(df["systolic_bp"], bins=20, color="skyblue", edgecolor="black")
plt.title("Hur ser blodtrycket ut i studien?")
plt.xlabel("Systoliskt blodtryck (mmHg)")
plt.ylabel("Antal personer")
plt.show()

In [ ]:
# Om vi gör en boxplot, kan vi jämföra vikt mellan män och kvinnor
# Blir det någon skillnad? Medianen, spridningen – allt syns på en gång.
# Video - Att välja rätt figur

plt.figure(figsize=(6, 6))
df.boxplot(column="weight", by="sex", grid=False, patch_artist=True,
           boxprops=dict(facecolor="lightgreen", color="black"),
           medianprops=dict(color="red"))
plt.title("Vikt – finns skillnad mellan män och kvinnor?")
plt.suptitle("")  # Tar bort undertitel
plt.xlabel("Kön")
plt.ylabel("Vikt (kg)")
plt.show()

In [ ]:
# Nu kollar vi hur många som röker – stapeldiagram med procent.
# Blir det en stor skillnad? Är det rimligt?
# Video - Matplotlib

smoker_counts = df["smoker"].value_counts(normalize=True)  # Andel
smoker_counts = smoker_counts[["Yes", "No"]]  # Yes kommer först

plt.figure(figsize=(6, 5))
smoker_counts.plot(kind="bar", color=["salmon", "lightgray"], edgecolor="black")
plt.title("Hur många röker i studien?")
plt.xlabel("Rökare")
plt.ylabel("Andel")
plt.xticks(rotation=0)
plt.ylim(0, 1)

# Lägger till procent ovanpå – det blir enkelt att läsa
for i, v in enumerate(smoker_counts):
    plt.text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")

plt.show()

## Simulering – hur kan slumpen ser ut?

Ska räkna först den riktiga andelen med sjukdom.  
Sen simulerar 1000 nya personer med samma chans – vad blir det då?  
Blir det nära verkligheten?

In [ ]:
# Räknar riktig andel med sjukdom (1 = ja)

disease_prop = df["disease"].mean()
print(f"Riktig andel med sjukdom: {disease_prop:.1%}")

# Simulerar 1000 nya personer med samma sannolikhet
# Video (sampling och variation)

simulated = np.random.binomial(n=1, p=disease_prop, size=1000)
simulated_prop = simulated.mean()

print(f"Simulerad andel: {simulated_prop:.1%}")
print(f"Skillnad: {abs(disease_prop - simulated_prop):.2%}")

## Konfidensintervall – hur säker är man på blodtrycket?

Räknar 95%-ish intervall för medelblodtrycket.  
Om intervallet är small -- då är jag säker. Om det e bredd -- mer data behövs

In [ ]:
# Räknar medel och standardavvikelse för blodtrycket
# Statistik

mean_bp = df["systolic_bp"].mean()
std_bp = df["systolic_bp"].std()
n = len(df)

print(f"Medelblodtryck: {mean_bp:.1f} mmHg")
print(f"Standardavvikelse: {std_bp:.1f}")
print(f"Antal personer: {n}")

# 95% konfidensintervall med normalapprox.
# Video (konfidensintervall)

ci = stats.norm.interval(0.95, loc=mean_bp, scale=std_bp / np.sqrt(n))
print(f"95% konfidensintervall: [{ci[0]:.1f}, {ci[1]:.1f}] mmHg")

## Hypotesprövning – har rökare högre blodtryck?

H₀: Rökare och icke-rökare har samma medelblodtryck.  
H₁: Rökare har högre medelblodtryck.  

Vi kör ett ensidigt t-test. Om p-värde < 0.05 -- förkastar vi H₀

In [ ]:
# Delar upp i rökare och icke-rökare
# Video (Rensa och sammanfatta)

smokers = df[df["smoker"] == "Yes"]["systolic_bp"]
non_smokers = df[df["smoker"] == "No"]["systolic_bp"]

print(f"Rökare: {len(smokers)} personer, medel = {smokers.mean():.1f} mmHg")
print(f"Icke-rökare: {len(non_smokers)} personer, medel = {non_smokers.mean():.1f} mmHg")

# Ensidigt t-test: rökare > icke-rökare?
# Video (Hypotesprövning)

t_stat, p_value = stats.ttest_ind(smokers, non_smokers, alternative="greater")

print(f"\nT-värde: {t_stat:.2f}")
print(f"P-värde (ensidigt): {p_value:.4f}")

# Slutsats

if p_value < 0.05:
    print("Förkastar H₀: Rökare har signifikant högre blodtryck!")
else:
    print("Kan inte förkasta H₀: Ingen tydlig skillnad (p ≥ 0.05)")